# T10 -- Causal Forest, D34 categorical-representation RESOURCE gate (K=32)

**Authorized scope, this notebook version:** run exactly one RESOURCE-gate attempt --
`categorical_k=32`, `population_size=2,000,000` -- against the already-predeclared
T10 RESOURCE cohort/protocol (`configs/t10_causal_forest.json`). It records
MEASURED_RESOURCE, a PROJECTED_FULL extrapolation to the D31 one-shot 9,785,714-row
FULL scale, and a `resource_gate_verdict` (memory/wall-clock/correctness/swap only --
never a Qini/AUUC/uplift/performance metric, never OTHER-bucket mass).

**What this notebook does NOT do:** it does not run `categorical_k=16` or `=8`
(those are separate, later, explicitly-authorized attempts if K=32 fails), it does
not run the D31 T11 FULL fit, and it never touches held-out data. All model/encoder
logic lives in `src/causal_forest_baseline.py` and `src/causal_forest_runner.py`;
this notebook only loads inputs, builds one `StageRequest`, calls the existing
`run_stage()`, and reports what it wrote -- it does not reimplement any of it.

**Environment.** Kaggle is the authoritative heavy-compute environment for this
run (owner directive, 2026-08-24 -- see `configs/t10_causal_forest.json`
`execution_environment`). This notebook is written to run unchanged on Kaggle or
locally: it locates the repository root from its own working directory exactly as
`notebooks/internal/t05_frozen_split.ipynb` does, and fails closed with a clear
error if a required input cannot be resolved rather than guessing a path.

In [ ]:
from __future__ import annotations

import json
import sys
from datetime import datetime, timezone
from pathlib import Path


def _find_repo_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / "src" / "data.py").is_file():
            return candidate
    raise RuntimeError(f"Could not locate repository root above {start}")


WORKING_DIRECTORY = Path.cwd().resolve()
REPO_ROOT = _find_repo_root(WORKING_DIRECTORY)
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

# Reuse the SAME loading helpers scripts/t11_run_stage.py's CLI path uses --
# never a second, notebook-local copy of dataset/split-loading logic.
from scripts.t11_run_stage import _load_dataset, _load_train_validation_ids

from src.causal_forest_baseline import CATEGORICAL_ENCODER_K_LADDER, FROZEN_CAUSAL_FOREST_CONFIG
from src.causal_forest_runner import (
    STAGES,
    CausalForestRunnerError,
    StageRequest,
    modeling_environment,
    run_stage,
    write_active_sentinel,
)

T10_CONFIG_PATH = REPO_ROOT / "configs" / "t10_causal_forest.json"
T05_CONFIG_PATH = REPO_ROOT / "configs" / "t05_split.json"

print("Repo root:", REPO_ROOT)


## 1. Verify the D34 representation decision and T10 lifecycle state

In [ ]:
t10_config = json.loads(T10_CONFIG_PATH.read_text(encoding="utf-8"))

EXPECTED_LIFECYCLE = "T10_REPRESENTATION_SELECTED_PENDING_RESOURCE_GATE"
if t10_config["lifecycle_state"] != EXPECTED_LIFECYCLE:
    raise CausalForestRunnerError(
        f"configs/t10_causal_forest.json lifecycle_state is {t10_config['lifecycle_state']!r}, "
        f"expected {EXPECTED_LIFECYCLE!r} -- this notebook version is only authorized to run "
        "against that exact state."
    )

d34 = t10_config["categorical_representation_D34"]
if tuple(d34["k_ladder"]) != CATEGORICAL_ENCODER_K_LADDER:
    raise CausalForestRunnerError(
        f"config K ladder {d34['k_ladder']} does not match the frozen code-level ladder "
        f"{CATEGORICAL_ENCODER_K_LADDER} -- config/code have drifted, stop."
    )
if "resource" not in STAGES:
    raise CausalForestRunnerError("src.causal_forest_runner.STAGES does not include 'resource'")

print("T10 lifecycle_state:", t10_config["lifecycle_state"])
print("D34 status:", d34["status"])
print("D34 K ladder (config == code):", d34["k_ladder"])
print("K selection rule:", d34["k_selection_rule"])
print("RESOURCE population declared in config:", t10_config["scale_gating"]["resource_gate_size"])


## 2. Resolve and verify dataset identity, T05 split, and package environment

Fails closed on a checksum mismatch, a non-`T05_SPLIT_ACCEPTED` state, a membership
hash mismatch, or an `econml` version drift from the frozen T10 config -- never
silently proceeds against an unexpected input.

In [ ]:
dataset, dataset_sha256 = _load_dataset(REPO_ROOT)
train_ids, validation_ids = _load_train_validation_ids(REPO_ROOT)

print(f"Processed dataset opened; processed_sha256={dataset_sha256}")
print(f"TRAIN ids: {len(train_ids):,}  VALIDATION ids: {len(validation_ids):,}")

RESOURCE_POPULATION_SIZE = t10_config["scale_gating"]["resource_gate_size"]
if RESOURCE_POPULATION_SIZE > len(train_ids):
    raise CausalForestRunnerError(
        f"resource_gate_size ({RESOURCE_POPULATION_SIZE:,}) exceeds available TRAIN ids "
        f"({len(train_ids):,})"
    )

environment = modeling_environment()  # raises on an econml version mismatch
print("econml:", environment["hard_match"]["econml"])
print("scikit-learn:", environment["hard_match"]["scikit_learn"])
print("numpy:", environment["hard_match"]["numpy"], " pandas:", environment["hard_match"]["pandas"])


## 3. Immutable run initialization -- run_id, run_root, active sentinel

In [ ]:
RUN_ID = datetime.now(timezone.utc).strftime("t10_resource_k32_%Y%m%dT%H%M%SZ_%f")
RUN_ROOT_BASE = REPO_ROOT / "outputs" / "runs"
RUN_ROOT = RUN_ROOT_BASE / RUN_ID
SENTINEL_PATH = RUN_ROOT_BASE / ".t11_resource_active_run_id.txt"

RUN_ROOT_BASE.mkdir(parents=True, exist_ok=True)
write_active_sentinel(SENTINEL_PATH, RUN_ID)  # atomic write+rename -- also proves writability

print("run_id:", RUN_ID)
print("run_root:", RUN_ROOT)
print("outputs/runs is writable:", RUN_ROOT_BASE.exists())


## 4. Build the StageRequest -- K=32 only, the frozen T10 RESOURCE protocol

- `stage="resource"`, `categorical_k=32` (first rung of the ladder -- never 16/8 here)
- `population_size=2,000,000` (the already-predeclared T10 RESOURCE cohort size)
- `sampling_seed=42`, `model_seed=42` -- the primary D31 seed, not a robustness seed
- `diagnostic_sample_size=100,000` -- the D31-approved bounded post-fit audit sample,
  applied identically at every stage (`configs/t11_causal_forest_full.json`)
- VALIDATION is **not** bounded: the complete frozen VALIDATION cohort is scored,
  exactly as `robustness`/`full` already do (never smoke's bounded convention) --
  this is what makes `project_full_scale_resource()`'s "VALIDATION-side quantities
  are already at full size" assumption correct for this run
- `config=FROZEN_CAUSAL_FOREST_CONFIG` unchanged -- no hyperparameter is touched
  for a RESOURCE gate

In [ ]:
DIAGNOSTIC_SAMPLE_SIZE = 100_000  # D31-approved, applied identically at every stage
CATEGORICAL_K = 32  # first rung only -- this notebook version does not authorize 16 or 8

request = StageRequest(
    stage="resource",
    run_id=RUN_ID,
    run_root=RUN_ROOT,
    dataset=dataset,
    dataset_sha256=dataset_sha256,
    train_ids=train_ids,
    validation_ids=validation_ids,          # complete frozen VALIDATION cohort, not bounded
    population_size=RESOURCE_POPULATION_SIZE,
    sampling_seed=42,
    model_seed=42,
    diagnostic_sample_size=DIAGNOSTIC_SAMPLE_SIZE,
    categorical_k=CATEGORICAL_K,
    config=FROZEN_CAUSAL_FOREST_CONFIG,
)

assert request.categorical_k == 32
assert request.population_size == 2_000_000
assert request.stage == "resource"
print("StageRequest built and validated (no fit performed yet).")
print(json.dumps({
    "stage": request.stage,
    "categorical_k": request.categorical_k,
    "population_size": request.population_size,
    "sampling_seed": request.sampling_seed,
    "model_seed": request.model_seed,
    "diagnostic_sample_size": request.diagnostic_sample_size,
}, indent=2))


## 5. EXECUTE -- real compute (K=32 RESOURCE fit + complete VALIDATION scoring)

This is the only cell in this notebook that performs real, resource-consuming
computation: it fits `econml.grf.CausalForest` on 2,000,000 TRAIN rows (encoded to
`8*(32+1)+4=268` columns by `CausalForestCategoricalEncoder`) and scores the
complete frozen VALIDATION cohort. It does not run until the flag below is
explicitly set to `True` -- mirroring `scripts/t11_run_stage.py`'s own
`--i-understand-this-executes-real-data-model-training` confirmation gate.

In [ ]:
I_UNDERSTAND_THIS_RUNS_REAL_COMPUTE = False  # set True immediately before running this cell

if not I_UNDERSTAND_THIS_RUNS_REAL_COMPUTE:
    raise RuntimeError(
        "Refusing to run: set I_UNDERSTAND_THIS_RUNS_REAL_COMPUTE = True above to confirm this "
        "is an authorized, owner-approved K=32 RESOURCE execution."
    )

result = run_stage(request)
print(json.dumps(result, indent=2, sort_keys=True))


## 6. RESOURCE-gate results -- MEASURED_RESOURCE, PROJECTED_FULL, verdict

Reads back exactly what `run_stage()` already wrote -- no recomputation, no second
source of truth.

In [ ]:
resource_evidence = json.loads((RUN_ROOT / "audit" / "resource_evidence.json").read_text(encoding="utf-8"))
model_serialized = json.loads(
    (RUN_ROOT / "audit" / "checkpoints" / "002_model_serialized.json").read_text(encoding="utf-8")
)
diagnostic_summary = json.loads(
    (RUN_ROOT / "audit" / "diagnostic_support_summary.json").read_text(encoding="utf-8")
)
manifest = json.loads((RUN_ROOT / "audit" / "artifact_manifest.json").read_text(encoding="utf-8"))

print("run status:", manifest["status"])
print("categorical_k:", resource_evidence["categorical_k"])
print("transformed_dimensionality:", resource_evidence["transformed_dimensionality"])
print("other_bucket_mass_by_feature:", resource_evidence["other_bucket_mass_by_feature"])
print("baseline_rss_bytes:", resource_evidence["baseline_rss_bytes"])
print("peak_rss_bytes (MEASURED):", resource_evidence["peak_rss_bytes"])
print("swap_used_bytes_max:", resource_evidence["swap_used_bytes_max"])
print("train_categorical_encoding_wall_seconds:", resource_evidence["train_categorical_encoding_wall_seconds"])
print("fit_wall_seconds (MEASURED):", resource_evidence["fit_wall_seconds"])
print("predict_wall_seconds (MEASURED):", resource_evidence["predict_wall_seconds"])
print()
print("--- PROJECTED_FULL (conservative linear extrapolation to 9,785,714 TRAIN rows) ---")
print(json.dumps(resource_evidence["projected_full_resource"], indent=2))
print()
print("--- resource_gate_verdict (resource/correctness only -- never a performance metric) ---")
print(json.dumps(resource_evidence["resource_gate_verdict"], indent=2))
print()
print("diagnostic_support passed (alpha/jac/tau finite):", diagnostic_summary["passed"])


## 7. Next step is an owner decision, not this notebook

If `resource_gate_verdict.passed` is `True`: K=32 is frozen as the D34 representation
width; T10 can proceed toward the correctness/leakage/honesty/validation/seed/
pre-test-freeze gates in `docs/adr/ADR-CF-implementation.md` at that K.

If `False`: retain this run's evidence (do not delete), and only then -- as a
separate, explicitly authorized run -- attempt `categorical_k=16` with a fresh
`run_id`. This notebook does not do that automatically, and K is never chosen by
comparing performance across attempts.

No held-out data was accessed. No K=16/K=8 attempt or T11 FULL fit occurred from
this notebook.